# Quantum latent distributions in deep generative modelsReproduction of [Bacarreza et al., arXiv:2508.19857](https://arxiv.org/abs/2508.19857) with [MerLin](https://merlinquantum.ai).In a GAN the generator is a **deterministic** map applied to a latent sample `z ~ P_z`. The latent is therefore the only place randomness enters, and its complexity bounds the complexity of everything the model can produce. The paper's Theorem 1 makes this precise: if `P_z` is in the class `Q` of distributions that are efficiently quantum-samplable but not classically samplable, and `g` is invertible with an efficiently classically implementable, Lipschitz-continuous inverse, then the pushforward `P_g(z)` is not in the classically samplable class `C`.Boson sampling supplies exactly such a `P_z`. Crucially the photonic circuit is **fixed and random** — no gradients ever flow into it — so the quantum device is used as a *sampler*, not as a trainable layer, and its cost is a one-off bank of samples drawn before training starts.This notebook walks through the pieces. The full studies are run through the shared runner:```bashpython ../../implementation.py --config configs/synthetic_datasets.json```

In [ ]:
import sysfrom pathlib import Pathsys.path.insert(0, str(Path.cwd()))import merlin as MLimport numpy as npimport torchfrom lib.circuits import haar_unitary, delay_line_unitary, to_circuitfrom lib.latents import build_latent, exact_distribution, sample_boson, sample_distinguishableprint("MerLin", ML.__version__)

## 1. Does the sampler sample the right thing?Perceval's Clifford & Clifford 2017 algorithm draws exact boson-sampling outcomes in polynomial time without ever forming the full distribution — essential, since 16 photons in 32 modes spans about 2.6e12 Fock states. We check it against MerLin's exact simulation on a system small enough to enumerate.**The one real trap.** `QuantumLayer` defaults to `ComputationSpace.UNBUNCHED`, which keeps only collision-free outcomes *and renormalises them to sum to 1*. It returns something that looks like a perfectly valid distribution while silently discarding every bunched event — and bunching is precisely where boson sampling's structure lives.

In [ ]:
from collections import Counterunitary = haar_unitary(6, np.random.default_rng(7))keys, exact = exact_distribution(unitary, 3)   # uses ComputationSpace.FOCKdef empirical(samples):    counts = Counter(map(tuple, samples.tolist()))    return np.array([counts.get(tuple(int(x) for x in k), 0) for k in keys]) / len(samples)for n in (12_500, 50_000, 200_000, 800_000):    tvd = 0.5 * np.abs(empirical(sample_boson(unitary, 3, n, seed=3)) - exact).sum()    print(f"N={n:>7}  TVD={tvd:.4f}")

In [ ]:
for space in (ML.ComputationSpace.UNBUNCHED, ML.ComputationSpace.FOCK):    layer = ML.QuantumLayer(        circuit=to_circuit(unitary),        input_state=[1, 1, 1, 0, 0, 0],        trainable_parameters=[], input_parameters=[],        measurement_strategy=ML.MeasurementStrategy.probs(space),    )    out = layer()    print(f"{space!s:<32} {out.shape[-1]:>3} states, mass={float(out.sum()):.4f}")

Both sum to 1. The `UNBUNCHED` one is a distribution over 20 of the 56 Fock states — for this system that is half the events thrown away.## 2. Why the distinguishable-photon control is the baseline that mattersRun the *same* interferometer with distinguishable photons and interference vanishes. Each photon then picks its output mode independently from `|U[:, k]|²`, so the single-mode marginals are unchanged — the control cannot be told apart from the boson sampler by any first-order statistic — while the joint distribution is completely different.Any gap between these two is attributable to multi-photon interference, and not to the latent merely being discrete, correlated or heavy-tailed. That is what makes the paper's comparison convincing.

In [ ]:
quantum = sample_boson(unitary, 3, 400_000, seed=3)classical = sample_distinguishable(unitary, 3, 400_000, seed=3)print("max marginal difference :", np.abs(quantum.mean(0) - classical.mean(0)).max().round(4))print("TVD of the joints       :", round(0.5 * np.abs(empirical(classical) - exact).sum(), 4))print("bunching rate  quantum  :", round(float((quantum.max(1) > 1).mean()), 3))print("bunching rate  classical:", round(float((classical.max(1) > 1).mean()), 3))

## 3. The four latent distributionsAll four subclass `merlin.LatentDistribution`, so they drop into anything MerLin expects a latent for — including `merlin.PhotonicGenerator`.Following Appendix C, banks are **centred** to zero mean and not rescaled: *"all distributions were centered to have a mean value of 0 before being injected into the generator."* That choice matters more than it looks, and `--normalize standardize` is available to test its effect.

In [ ]:
latents = {    kind: build_latent(kind, dim=16, seed=0, architecture="1-3-9", bank_size=100_000)    for kind in ("gaussian", "bernoulli", "distinguishable", "boson")}for name, latent in latents.items():    z = latent.sample(20_000).numpy()    corr = np.corrcoef(z.T)    off_diagonal = corr[~np.eye(16, dtype=bool)]    print(f"{name:<16} std={z.std():.3f}  max|z|={np.abs(z).max():5.2f}  "          f"max|corr|={np.abs(off_diagonal).max():.3f}")

Two things to read off that table. The classical baselines are **factorisable** — correlations at noise level — while the photonic ones are not; the paper proposes exactly this as the mechanism behind the mixture-of-Gaussians result. And the boson sampler's `max|z|` sits well above its distinguishable control: that is bunching, and it is the main practical difficulty in training with these latents.## 4. Training a GAN on the latentNothing in the training loop is quantum. The generator uses the paper's design constraint — non-decreasing hidden widths, so the map is invertible-in-principle and Theorem 1 applies — with an affine copy of `z` re-injected at every layer.

In [ ]:
from lib.datasets import quantum_datasetfrom lib.gan import Critic, Generator, WGANGPConfig, train_wgan_gpfrom lib.metrics import l1_to_nearest_integerdata = torch.from_numpy(quantum_dataset(20_000, seed=0))   # 8 photons, 16 modeslatent = build_latent("boson", dim=16, seed=0, bank_size=50_000)generator = Generator(16, 16, hidden=(512, 512), reinject=True)critic = Critic(16, hidden=(512, 512))train_wgan_gp(data, latent, generator, critic,              WGANGPConfig(iterations=300, n_critic=5, batch_size=128))   # demo budgetwith torch.no_grad():    fake = generator(latent.sample(5_000))print("L1 to nearest integer:", round(l1_to_nearest_integer(fake), 4))

Three hundred iterations is a demo, not a result. The real comparison is `configs/synthetic_datasets.json`, which reproduces the paper's Table I; see the README for the numbers obtained.## 5. Swapping in real hardwareBecause every latent is routed through a pre-drawn bank, moving from simulation to a QPU changes one line and nothing downstream:```pythonfrom lib.hardware import hardware_latentlatent = hardware_latent("qpu:belenos", n_modes=32, n_photons=16,                         n_samples=500_000, architecture="1-1")```or from the CLI:```bashpython ../../implementation.py --config configs/synthetic_datasets.json \    --latent-source hardware --platform qpu:belenos --latent-dim 32````"1-1"` is the delay-line configuration of the ORCA PT-2 the paper used — 16 photons in 32 channels, half a million samples in 40 minutes. Lossy shots are kept rather than post-selected: loss is part of the distribution the generator actually sees, and discarding it biases the latent.## 6. Where to go nextThe paper deliberately freezes the photonic circuit. MerLin makes two extensions cheap:- build the latent from a `QuantumLayer` with trainable phases and learn it jointly with the generator;- sweep `pcvl.NoiseModel(indistinguishability=...)` from 0 to 1, which interpolates *continuously* between the distinguishable control and the quantum latent instead of comparing only the two endpoints.